### Sonar signal classification

**Dataset description:**

The Sonar dataset contains sonar signal readings collected by bouncing sound waves off objects underwater, with the goal of predicting whether the object is a rock or a mine. Each row represents one sonar return, described using numerical energy measurements from multiple frequencies.

The dataset has 60 numerical feature columns that represent the strength of sonar signals at different frequencies. The final column is the target label, where R denotes rock and M denotes mine.

### 1. Setup

In [ ]:
# import the dependencies
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
# configurations
pd.set_option("display.max_columns", None)
sns.set_theme(style="darkgrid")


### 2. Load Data

In [ ]:
# read csv file into a pandas dataframe
df = pd.read_csv("sonar_data.csv", header=None)

In [ ]:
df.head()

In [ ]:
df.shape

### 3. Exploratory Data Analysis (EDA)

In [ ]:
# Basic data overview
df.info()

In [ ]:
# missing value analysis
df.isnull().sum()

In [ ]:
df.columns

In [ ]:
# identify the presence of encoded missing values
for col in df.columns:
    print(df[col].value_counts())

In [ ]:
# target distribution 
print(df[60].value_counts())
print("-" * 40)
print(df[60].value_counts(normalize=True)*100)

In [ ]:
# map labels
df[60] = df[60].map({"M":1, "R":0})

In [ ]:
# target distribution
print(df[60].value_counts())
print("-" * 50)
print(df[60].value_counts(normalize=True)*100)

In [ ]:
df.head()

In [ ]:
# identify duplicate rows
duplicate_mask = df.duplicated()
num_duplicates = duplicate_mask.sum()

print("number of duplicate rows", num_duplicates)
print("duplicated rows:\n",df[duplicate_mask])


In [ ]:
# if you want to duplicate
#df_no_duplicates = df.drop_duplicates()
#print("shape after dropping duplicates:", df_no_duplicates)

In [ ]:
# constant and quasi-constant columns - can identify the columns that has almost similar values throughout

n_rows = len(df)
nunique = df.nunique()

constant_cols = nunique[nunique ==1].index.tolist()
print("constant columns:", constant_cols)


# quasi constant : top value more than 95 percent 
quasi_constant_cols = []

for col in df.columns:
    top_freq = df[col].value_counts(normalize=True, dropna=False).values[0]
    if top_freq > 0.95 and col not in constant_cols:
        quasi_constant_cols.append(col)

print("quasi constant columns (top value more than 95 percent):", quasi_constant_cols)

In [ ]:
# group the data based on mean of features for class 1 and class 0 (0 - Rock, 1-Mine)
df.groupby(60).mean()

In [ ]:
# descriptive stat
df.describe()

Data Visualization

In [ ]:
# distribution histogram plot
#for col in df.columns:
 #   plt.figure(figsize=(4,3))
  #  sns.histplot(df[col], kde = True)
    # plt.title(col)
    #plt.xlabel(col)
    #plt.ylabel("frequency")
    #plt.show()

In [ ]:
# subplots
fig, axes = plt.subplots(13,5, figsize = (15, 18))
axes = axes.flatten()

for i, col in enumerate(df.columns):
    sns.histplot(df[col], kde = True, ax = axes[i])
    axes[i].set_title(col, fontsize = 8)

plt.tight_layout()
plt.show()

In [ ]:
# outliers - boxplot
fig, axes = plt.subplots(13,5, figsize = (15, 20))
axes = axes.flatten()

for i, col in enumerate(df.columns):
    sns.boxplot(x = df[col], ax =axes[i])
    axes[i].set_title(col, fontsize = 8)

plt.tight_layout()
plt.show()

**NOTE: We cannot create a pairplot for numeric vs numeric columns as it would create a 60 x 60 plot. We can create correlation heatmap to underatnd the relationship better (in training data)**

In [ ]:
# understand features relationship
plt.figure(figsize=(14,10))
sns.heatmap(df.corr(), cmap = "coolwarm", center=0)
plt.title("correlation heatmap")
plt.show()

**EDA Insights:**
- The dataset has lesser number of rows (208), so, we can use a simplr model
- High number of numerical features
- balanced dataset (R & M)
- no missing values, no constant or quasi constant columns, no id column
- mean groupby reveals clear difference between the two classes
- most columns are right skewed
- some outliers are present but they can be valid values
- some features show correlation but not in 0.9 or 0.95 level

### 4. Data Preprocessing

In [ ]:
# separate features and target
X = df.drop(columns=[60])
y = df[60]

In [ ]:
X.head()

In [ ]:
y.head()

In [ ]:
# train test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
print("dataset shape:", X.shape)
print("training dataset shape:", X_train.shape)
print("test dataset shape:", X_test.shape)

In [ ]:
# feature scalling
scaler = StandardScaler()

In [ ]:
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### 5. Model training

:**Support Vector Classifier (SVC) works well for high dimensional data**

In [ ]:
model = SVC()

In [ ]:
# train the model
model.fit(X_train_scaled, y_train)

### 6. Model Evaluation

In [ ]:
y_train_pred = model.predict(X_train_scaled)
train_acc = accuracy_score(y_train, y_train_pred)
print(f"Training data Accuracy: {round(train_acc * 100,2)}%")

In [ ]:
y_test_pred = model.predict(X_test_scaled)
test_acc = accuracy_score(y_test, y_test_pred)
print(f"Test data Accuracy: {round(test_acc * 100,2)}%")

In [ ]:
print("Training data - classification report")
print(classification_report(y_train, y_train_pred))

In [ ]:
print("Test data - classification report")
print(classification_report(y_test, y_test_pred))

### 7. Building a predictive system

In [ ]:
def predict_object(input_features):
    # scale features
    scaled_features = scaler.transform([input_features])
    # get prediction from teh model
    prediction = model.predict(scaled_features)
    print("Model prediction", prediction)
    if prediction[0] == 1:
        print("The object is identified as Mine 💣")
    else:
        print("The object is identified as Rock 🪨")


In [ ]:
X_test.head()

In [ ]:
y_test.head()

In [ ]:
# sample prediction get the value from X_test index
test_1 = X_test.loc[103].tolist()
print(test_1)

In [ ]:
predict_object(test_1)

In [ ]:
test_2 = X_test.loc[90].tolist()
print(test_2)

In [ ]:
predict_object(test_2)